# Analyze the Kaggle training run

This notebook reads the artifacts produced by `examples/kaggle_training.ipynb`
(downloaded from Kaggle into `data/`) and analyzes them: which history window
won and why, where the model beats the baselines, how well-calibrated the
prediction intervals are, what drives the forecasts, and how the error breaks
down by season and asset.

It does **no fetching** and **no Kaggle run**, it only reads local files in
`data/` (see `data/README.md` for the expected file list). Sections that need
a held-out test set retrain a model locally on the downloaded panel, which is
fast and offline.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from climagrid.forecasting.backtest import rolling_origin_splits
from climagrid.forecasting.dataset import build_supervised_frame
from climagrid.forecasting.models import LightGBMForecaster

warnings.filterwarnings("ignore")

# Find the data directory (run from repo root or from examples/).
CANDIDATES = [Path("data"), Path("../data"), Path(".")]
DATA_DIR = next((p for p in CANDIDATES if (p / "manifest.json").exists()), Path("data"))
print("data dir:", DATA_DIR.resolve())

manifest = {}
if (DATA_DIR / "manifest.json").exists():
    manifest = json.loads((DATA_DIR / "manifest.json").read_text())
    print("manifest:", manifest)
else:
    print("No manifest.json found. Put the Kaggle outputs in", DATA_DIR, "and re-run.")

In [ ]:
def load_csv(name: str) -> pd.DataFrame | None:
    path = DATA_DIR / name
    if not path.exists():
        print("MISSING:", path)
        return None
    return pd.read_csv(path)

comparison = load_csv("model_comparison.csv")
scores = load_csv("backtest_scores.csv")

panel = None
panel_file = manifest.get("panel_file", "daily_panel_full.parquet")
if (DATA_DIR / panel_file).exists():
    panel = pd.read_parquet(DATA_DIR / panel_file)
    panel["date"] = pd.to_datetime(panel["date"])

print("comparison:", None if comparison is None else comparison.shape)
print("scores:    ", None if scores is None else scores.shape)
print("panel:     ", None if panel is None else panel.shape)

## 1. Which window won

The training notebook picked the most accurate window by lowest mean
out-of-sample MAE on the shared recent test period.

In [ ]:
best = manifest.get("best_window")
print("Most accurate window:", best)
comparison if comparison is None else comparison.set_index(comparison.columns[0])

## 2. Skill vs baselines, by horizon

`skill = 1 - MSE_model / MSE_baseline`: positive means the model beats the
baseline, zero is a tie, negative means it loses. Persistence is the hard
baseline at short lead times; climatology is the one to beat at longer ones.

In [ ]:
if scores is not None:
    windows = list(scores["window"].unique())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for metric, ax in zip(["skill_vs_persistence", "skill_vs_climatology"], axes):
        for w in windows:
            by_h = scores[scores["window"] == w].groupby("horizon_day")[metric].mean()
            ax.plot(by_h.index, by_h.values, marker="o", label=w)
        ax.axhline(0.0, color="grey", ls="--", lw=0.8)
        ax.set_title(metric)
        ax.set_xlabel("horizon (days)")
        ax.legend()
    axes[0].set_ylabel("skill score")
    fig.tight_layout()
    plt.show()

## 3. Accuracy by horizon (best window vs baselines)

Absolute error of the model's point forecast (`p50`) against the persistence
and climatology baselines, horizon by horizon.

In [ ]:
if scores is not None and best is not None:
    bw = scores[scores["window"] == best].groupby("horizon_day")[
        ["mae", "mae_persistence", "mae_climatology"]
    ].mean()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(bw.index, bw["mae"], marker="o", label="model (p50)")
    ax.plot(bw.index, bw["mae_persistence"], marker="s", label="persistence")
    ax.plot(bw.index, bw["mae_climatology"], marker="^", label="climatology")
    ax.set_title(f"MAE by horizon ({best} model)")
    ax.set_xlabel("horizon (days)")
    ax.set_ylabel("mean absolute error")
    ax.legend()
    fig.tight_layout()
    plt.show()
    bw.round(5)

## 4. Interval calibration

The `p10`-`p90` band is meant to be an 80% prediction interval, so coverage
(the fraction of actuals that fall inside it) should sit near **0.80**.
Coverage well below 0.80 means the intervals are too narrow / overconfident.
This is the gap a later calibration step (conformal prediction) would close.

In [ ]:
if scores is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for w in scores["window"].unique():
        cov = scores[scores["window"] == w].groupby("horizon_day")["interval_coverage"].mean()
        ax.plot(cov.index, cov.values, marker="o", label=w)
    ax.axhline(0.80, color="red", ls="--", lw=1.0, label="target 0.80")
    ax.set_ylim(0, 1)
    ax.set_title("80% interval coverage by horizon")
    ax.set_xlabel("horizon (days)")
    ax.set_ylabel("coverage")
    ax.legend()
    fig.tight_layout()
    plt.show()
    print("mean coverage per window:")
    print(scores.groupby("window")["interval_coverage"].mean().round(3))

## 5. What drives the forecasts (feature importance)

LightGBM importance (split gain) for the median (`p50`) models, averaged
across horizons, for the winning model. Tells us whether the model leans on
recent autocorrelation (lags / rolling stats), seasonality (day-of-year), or
location.

In [ ]:
best_model = None
best_file = manifest.get("best_model_file")
if best_file and (DATA_DIR / best_file).exists():
    best_model = LightGBMForecaster.load(DATA_DIR / best_file)
    cfg = best_model._config
    median_q = 0.5
    imps = [
        best_model._models[(h, median_q)].feature_importances_
        for h in range(1, cfg.horizon_days + 1)
        if (h, median_q) in best_model._models
    ]
    fi = pd.Series(np.mean(imps, axis=0), index=best_model._predictors).sort_values()
    fig, ax = plt.subplots(figsize=(7, 5))
    fi.plot.barh(ax=ax)
    ax.set_title(f"Feature importance (p50 models, {manifest.get('best_window')})")
    ax.set_xlabel("mean split gain")
    fig.tight_layout()
    plt.show()
else:
    print("Best model file not found in", DATA_DIR)

## 6. Error patterns by season and asset (held-out recompute)

The saved model was trained on the whole panel, so it has no held-out set of
its own. To inspect honest out-of-sample residuals we retrain locally on a
train split of the downloaded panel and score the most recent 90 days. No
fetching is involved.

In [ ]:
resid = None
if panel is not None and best_model is not None:
    cfg = best_model._config
    target = cfg.targets[0]
    sup = build_supervised_frame(panel, target, cfg)
    dates = [pd.Timestamp(d) for d in sup["date"].unique()]
    splits = rolling_origin_splits(dates, cfg, n_splits=1, test_size_days=90)
    train_dates, test_dates = splits[-1]
    train = sup[sup["date"].isin(train_dates)]
    test = sup[sup["date"].isin(test_dates)]
    holdout_model = LightGBMForecaster(cfg).fit(train, target)
    preds = holdout_model.predict(test, target)

    parts = []
    for h in range(1, cfg.horizon_days + 1):
        actual = test[["asset_id", "date", f"y_h{h}"]].dropna()
        ph = preds[preds["horizon_day"] == h][["asset_id", "origin_date", "forecast_date", "p50"]]
        merged = actual.merge(ph, left_on=["asset_id", "date"], right_on=["asset_id", "origin_date"])
        merged["abs_err"] = (merged["p50"] - merged[f"y_h{h}"]).abs()
        parts.append(merged[["asset_id", "forecast_date", "abs_err"]])
    resid = pd.concat(parts, ignore_index=True)
    resid["month"] = resid["forecast_date"].dt.month
    print("held-out test rows:", len(resid))
else:
    print("Need both the panel and the best model to recompute residuals.")

In [ ]:
if resid is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    by_month = resid.groupby("month")["abs_err"].mean()
    axes[0].bar(by_month.index, by_month.values)
    axes[0].set_title("Mean abs error by month (seasonality)")
    axes[0].set_xlabel("month")
    axes[0].set_ylabel("MAE")

    by_asset = resid.groupby("asset_id")["abs_err"].mean().sort_values()
    top = by_asset.tail(15)  # the 15 hardest-to-forecast assets
    axes[1].barh(range(len(top)), top.values)
    axes[1].set_yticks(range(len(top)))
    axes[1].set_yticklabels(top.index, fontsize=7)
    axes[1].set_title("Hardest assets (highest MAE)")
    axes[1].set_xlabel("MAE")
    fig.tight_layout()
    plt.show()

## Summary

Use this notebook to decide:
- **Which history window** to ship (Section 1, 3) and whether more years
  actually helped.
- **Where the model earns its keep** vs persistence/climatology (Section 2).
- **How trustworthy the intervals are** (Section 4): the coverage gap below
  0.80 is the target for the planned conformal-calibration step.
- **What the model relies on** (Section 5) and **where it struggles**, by
  season and asset (Section 6).

Calibration comes next; this analysis tells us how big the gap is and whether
the point forecasts are already good enough to build on.